# Regression and classification answer different questions

This notebook uses one simulated signal to predict both a **numerical return** and the **event that the return is positive**. The point is not to find a winning model; it is to see that an estimated magnitude and an estimated probability are different outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, brier_score_loss

rng = np.random.default_rng(7)
x = rng.normal(size=240)
future_return = 0.02 * x + rng.normal(0, 0.03, len(x))
up = (future_return > 0).astype(int)
train = np.arange(180)
test = np.arange(180, 240)

reg = LinearRegression().fit(x[train, None], future_return[train])
clf = LogisticRegression().fit(x[train, None], up[train])
return_hat = reg.predict(x[test, None])
up_probability = clf.predict_proba(x[test, None])[:, 1]

print(f'Regression test MSE: {mean_squared_error(future_return[test], return_hat):.5f}')
print(f'Classification test Brier score: {brier_score_loss(up[test], up_probability):.3f}')
print(f'At signal = 1: predicted return = {reg.predict([[1]])[0]:.3%}, probability(up) = {clf.predict_proba([[1]])[0, 1]:.1%}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(x[test], future_return[test], alpha=.65, label='observed')
order = np.argsort(x[test])
axes[0].plot(x[test][order], return_hat[order], color='firebrick', label='regression')
axes[0].set(title='Magnitude', xlabel='signal', ylabel='future return')
axes[0].legend()
axes[1].scatter(x[test], up[test], alpha=.45, label='event')
axes[1].plot(x[test][order], up_probability[order], color='teal', label='probability')
axes[1].set(title='Event probability', xlabel='signal', ylabel='P(return > 0)')
axes[1].legend(); plt.tight_layout();

## What to notice
The models see the same input, but one returns a number in return units while the other returns an event probability. A trading threshold or position size is a **third** object that must be chosen after the prediction.